# Pedro y el Lobo: Introduccion al Aprendizaje por Refuerzo

## Que aprendemos aqui?

1. **Que es el Aprendizaje por Refuerzo** (RL) y como funciona
2. **El entorno**: un tablero con suelo, agua, arboles, manzanas y lobos
3. **Acciones**: arriba, abajo, izquierda, derecha
4. **Politica**: la estrategia que sigue el agente
5. **Recompensa**: como motivar al agente a tomar buenas decisiones
6. **Q-Learning**: el algoritmo que aprende la mejor estrategia
7. **Exploracion vs Explotacion**: el balance clave del RL

## Imports

Necesitamos:
- `matplotlib`: para graficar
- `numpy`: para operaciones matematicas
- `random`: para elecciones aleatorias
- `math`: para funciones matematicas

In [ ]:
import sys
!{sys.executable} -m pip install opencv-python
sys.path.append('..')
import matplotlib.pyplot as plt
import numpy as np
import random
import math

## El entorno: Pedro y el Lobo

### Que es el Aprendizaje por Refuerzo?

El **Aprendizaje por Refuerzo** (RL) es una tecnica que permite a un **agente** aprender a comportarse en un **entorno** ejecutando muchos experimentos.

**Analogia**: Imagina que ensenas a un perro a hacer trucos.
- **Agente**: el perro
- **Entorno**: la casa, el jardin
- **Acciones**: sentarse, dar la pata, saltar
- **Recompensa**: premio (comida) cuando hace algo bien
- **Castigo**: nada cuando hace algo mal

Con el tiempo, el perro aprende que acciones le dan mas premios.

### Nuestro entorno

Pedro necesita encontrar una manzana en un tablero. El tablero tiene:
- **Suelo**: donde Pedro puede caminar
- **Agua**: donde no puede caminar (se ahoga)
- **Arboles/hierba**: donde puede descansar
- **Manzana**: su objetivo
- **Lobo**: peligro que debe evitar

### Crear el tablero

Usamos un modulo externo `rlboard` que contiene la clase `Board` para manejar el tablero.

In [ ]:
from rlboard import *

width, height = 8, 8
m = Board(width, height)
m.randomize(seed=13)
m.plot()

## Acciones y Politica

### Acciones disponibles

En cada posicion, Pedro puede elegir entre 4 acciones:

| Accion | Movimiento | Codigo |
|--------|------------|--------|
| U (Up) | Arriba | (0, -1) |
| D (Down) | Abajo | (0, 1) |
| L (Left) | Izquierda | (-1, 0) |
| R (Right) | Derecha | (1, 0) |

**Nota**: El eje Y va hacia abajo en las imagenes, por eso -1 es arriba.

### Que es una Politica?

Una **politica** es la estrategia que sigue el agente para elegir acciones.

**Ejemplo simple**: Politica aleatoria
- En cada paso, elige una accion al azar
- No aprende nada
- Es como un bebe gateando sin rumbo fijo

In [ ]:
actions = { "U" : (0,-1), "D" : (0,1), "L" : (-1,0), "R" : (1,0) }
action_idx = { a : i for i,a in enumerate(actions.keys()) }

print("Acciones disponibles:", actions)
print("Indice de acciones:", action_idx)

## Camino Aleatorio

Primero implementemos la estrategia mas simple: **elegir acciones al azar**.

### Funcion walk

Esta funcion mueve a Pedro por el tablero hasta que:
- **Encuentra la manzana**: exito! (devuelve numero de pasos)
- **Se cae al agua o encuentra al lobo**: fracaso! (devuelve -1)

In [ ]:
def random_policy(m):
    """Politica aleatoria: elige una accion al azar"""
    return random.choice(list(actions))

def walk(m, policy, start_position=None):
    """Mueve a Pedro por el tablero segun la politica dada"""
    n = 0
    
    if start_position:
        m.human = start_position 
    else:
        m.random_start()
    
    while True:
        if m.at() == Board.Cell.apple:
            return n
        if m.at() in [Board.Cell.wolf, Board.Cell.water]:
            return -1
        while True:
            a = actions[policy(m)]
            new_pos = m.move_pos(m.human, a)
            if m.is_valid(new_pos) and m.at(new_pos) != Board.Cell.water:
                m.move(a)
                break
        n += 1

walk(m, random_policy)

### Estadisticas del camino aleatorio

Ejecutamos el camino 100 veces y vemos:
- **Longitud promedio**: cuantos pasos en promedio hasta encontrar la manzana
- **Veces comido**: cuantas veces el lobo se lo comio

La longitud promedio sera de ~30-40 pasos, siendo que la distancia optimal es ~5-6 pasos. El camino aleatorio es MUY ineficiente.

In [ ]:
def print_statistics(policy):
    s, w, n = 0, 0, 0
    for _ in range(100):
        z = walk(m, policy)
        if z < 0:
            w += 1
        else:
            s += z
            n += 1
    print(f"Average path length = {s/n}, eaten by wolf: {w} times")

print_statistics(random_policy)

## Funcion de Recompensa

Para hacer la politica mas inteligente, necesitamos definir que movimientos son "mejores".

### Recompensas:

| Tipo | Recompensa | Por que? |
|------|------------|----------|
| Moverse | -0.1 | Penalizacion leve por cada paso |
| Encontrar manzana | +10 | Objetivo logrado! |
| Morir (lobo/agua) | -10 | Castigo fuerte |

**Clave**: La recompensa es **diferida**. No sabes si un movimiento fue bueno hasta que llegas al final. Es como un examen: no sabes si estudiaste bien hasta que ves las notas.

In [ ]:
move_reward = -0.1
goal_reward = 10
end_reward = -10

def reward(m, pos=None):
    pos = pos or m.human
    if not m.is_valid(pos):
        return end_reward
    x = m.at(pos)
    if x == Board.Cell.water or x == Board.Cell.wolf:
        return end_reward
    if x == Board.Cell.apple:
        return goal_reward
    return move_reward

## Q-Learning

### Que es Q-Learning?

Q-Learning es un algoritmo que aprende una **Q-Table** (tabla Q). Esta tabla guarda que tan buena es cada accion en cada estado.

**Analogia**: Imagina un mapa de Google Maps donde cada interseccion tiene flechas que indican "esta direccion es buena" o "esta direccion es mala".

### La Q-Table

- **Dimensiones**: ancho x alto x 4 (4 acciones)
- **Valores**: que tan buena es cada accion en esa posicion
- **Inicializacion**: todos iguales (0.25) = politica aleatoria

In [ ]:
Q = np.ones((width, height, len(actions)), dtype=float) * 1.0 / len(actions)

print("Forma de la Q-Table:", Q.shape)
print("Valores iniciales:", Q[0,0])

## Ecuacion de Bellman

La formula clave de Q-Learning:

`Q(s,a) = (1-α) * Q(s,a) + α * (r + γ * max Q(s',a'))`

Donde:
- `Q(s,a)`: valor de la accion `a` en el estado `s`
- `α` (alpha): tasa de aprendizaje (que tan rapido aprende)
- `r`: recompensa obtenida
- `γ` (gamma): factor de descuento (importancia del futuro)
- `max Q(s',a')`: mejor valor posible en el siguiente estado

**En lenguaje simple**: El valor de una accion = (lo que ya se) + (aprendizaje) * (recompensa + futuro)

Es como decir: "Esta accion vale lo que ya sabia mas un poco de lo que aprendi ahora."

## Exploracion vs Explotacion

El dilema fundamental del RL:

| Exploracion | Explotacion |
|-------------|-------------|
| Probar cosas nuevas | Usar lo que ya se |
| "Y si hay algo mejor?" | "Esto funciona, lo uso" |
| Buscar manzanas lejanas | Ir directo a la manzana conocida |

**Solucion**: Balance entre ambas. Al principio exploras mucho, despues explotas mas.

In [ ]:
def probs(v, eps=1e-4):
    """Convierte valores Q a probabilidades"""
    v = v - v.min() + eps
    v = v / v.sum()
    return v

## Algoritmo de Aprendizaje

### Pasos del algoritmo:

1. Inicializar Q-Table con valores iguales
2. Repetir 5000 veces (epochs):
   a. Empezar en posicion aleatoria
   b. En cada paso:
      - Elegir accion usando probabilidades de la Q-Table
      - Moverse y obtener recompensa
      - Actualizar Q-Table con ecuacion de Bellman
      - Si muere o llega a la manzana, terminar
3. La Q-Table ahora contiene la mejor estrategia

**Detalle tecnico**: Alpha empieza en 1 y disminuye exponencialmente. Asi aprendemos rapido al principio y refinamos despues.

In [ ]:
from IPython.display import clear_output

lpath = []

for epoch in range(5000):
    clear_output(wait=True)
    print(f"Epoch = {epoch}", end='')

    m.random_start()
    
    n = 0
    cum_reward = 0
    while True:
        x, y = m.human
        v = probs(Q[x, y])
        a = random.choices(list(actions), weights=v)[0]
        dpos = actions[a]
        m.move(dpos, check_correctness=False)
        r = reward(m)
        cum_reward += r
        if r == end_reward or cum_reward < -1000:
            lpath.append(n)
            break
        alpha = np.exp(-n / 3000)
        gamma = 0.5
        ai = action_idx[a]
        Q[x, y, ai] = (1 - alpha) * Q[x, y, ai] + alpha * (r + gamma * Q[x + dpos[0], y + dpos[1]].max())
        n += 1

### Visualizar la Q-Table aprendida

Las flechas ahora apuntan hacia la manzana! El agente aprendio las rutas optimas.

In [ ]:
m.plot(Q)

## Verificando la Politica

La Q-Table enumera la "atractividad" de cada accion en cada estado. Podemos usarla para navegar eficientemente.

### Politica estricta (siempre elegir la mejor accion)

En el caso mas simple, elegimos la accion con el mayor valor Q.

In [ ]:
def qpolicy_strict(m):
    x, y = m.human
    a = list(actions)[np.argmax(Q[x, y])]
    return a

walk(m, qpolicy_strict)

> **Problema**: A veces la politica estricta se "cuelga" porque dos estados se senalan mutuamente. Pedro fica dando vueltas entre dos casillas.

### Politica suave (muestrear segun probabilidades)

Una mejor estrategia es elegir acciones con probabilidad proporcional a los valores Q. Esto permite explorar mientras sigue la mejor ruta.

In [ ]:
def qpolicy(m):
    x, y = m.human
    v = probs(Q[x, y])
    a = random.choices(list(actions), weights=v)[0]
    return a

print_statistics(qpolicy)

**Resultado**: La longitud promedio baja de ~35 a ~3-6 pasos. Q-Learning funciona!

## Investigando el Proceso de Aprendizaje

In [ ]:
plt.plot(lpath)
plt.xlabel("Epoch")
plt.ylabel("Pasos")
plt.title("Longitud del camino durante el entrenamiento")
plt.show()

### Que vemos en la grafica?

- **Al principio**: la longitud INCREMENTA. El agente explora y se pierde mucho.
- **Despues**: la longitud DECREMENTA. Aprende rutas optimas.
- **Saltos repentinos**: la naturaleza estocastica del proceso. A veces "dania" la Q-Table con nuevos valores.

**Clave**: El exito depende de los **hiperparametros**:
- `alpha` (tasa de aprendizaje): que tan rapido aprende
- `gamma` (factor de descuento): importancia del futuro
- Balance entre exploracion y explotacion

## Conceptos Clave

| Concepto | Que es | Analogia |
|----------|--------|----------|
| **Agente** | Quien toma decisiones | Pedro |
| **Entorno** | Donde actua el agente | El tablero |
| **Estado** | Donde esta el agente ahora | Posicion en el tablero |
| **Accion** | Que puede hacer el agente | Arriba, abajo, izq, der |
| **Recompensa** | Que tan buena fue la accion | +10 manzana, -10 muerte |
| **Politica** | Estrategia para elegir acciones | "Siempre ir a la derecha" |
| **Q-Table** | Memoria de que funciona | Mapa con flechas |
| **Alpha** | Que tan rapido aprende | Velocidad de aprendizaje |
| **Gamma** | Importancia del futuro | Que tanto piensa en el futuro |

## Errores Comunes

| Error | Consecuencia |
|-------|--------------|
| Olvidar `sys.path.append('..')` | No encuentra rlboard |
| Alpha muy alto | Aprendizaje inestable |
| Alpha muy bajo | Aprende muy lento |
| Sin exploracion | Se queda en optima local |
| Epochs muy pocos | No aprende lo suficiente |
| Epochs muy muchos | Sobreajuste, desperdicia tiempo |

## Siguiente paso

En la **Leccion 2**, veremos como aplicar Q-Learning a un problema con **estados continuos**: CartPole. Aprenderemos a equilibrar un palo usando OpenAI Gym.